In [1]:
pip install arch

  Using cached arch-8.0.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
Using cached arch-8.0.0-cp313-cp313-win_amd64.whl (929 kB)
Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl (9.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

from statsmodels.tsa.stattools import adfuller
from arch.unitroot import PhillipsPerron

In [3]:
data_processed = pd.read_csv('../data/processed/data_processed.csv')

In [4]:
unit_root_vars = [
    "ln_FDI",
    "ln_TB",
    "ln_EX",
    "ln_IM",
    "ln_RER_pos",
    "ln_RER_neg",
    "IPI_VN",
    "IPI_World_diff",
    "ln_WTI",
    "ln_M2_diff",
    "COVID"
]

In [5]:
# 1. HÀM CHẠY ADF TEST
# ============================================================

def run_adf(series, regression="c", autolag="AIC"):
    """
    ADF Test
    H0: Chuỗi có unit root, tức là không dừng.
    Nếu p-value < 0.05: bác bỏ H0 => chuỗi dừng.

    regression:
    "c"  = có intercept
    "ct" = có intercept và trend
    "n"  = không intercept
    """

    y = series.dropna()

    if len(y) < 10:
        return np.nan, np.nan, np.nan, "Too few observations"

    try:
        result = adfuller(y, regression=regression, autolag=autolag)
        stat = result[0]
        pvalue = result[1]
        lag = result[2]
        return stat, pvalue, lag, "OK"
    except Exception as e:
        return np.nan, np.nan, np.nan, str(e)

In [6]:
# ============================================================
# 2. HÀM CHẠY PHILLIPS-PERRON TEST
# ============================================================

def run_pp(series, trend="c"):
    """
    Phillips-Perron Test
    H0: Chuỗi có unit root, tức là không dừng.
    Nếu p-value < 0.05: bác bỏ H0 => chuỗi dừng.

    trend:
    "c"  = có intercept
    "ct" = có intercept và trend
    "n"  = không intercept
    """

    y = series.dropna()

    if len(y) < 10:
        return np.nan, np.nan, np.nan, "Too few observations"

    try:
        pp = PhillipsPerron(y, trend=trend)
        stat = pp.stat
        pvalue = pp.pvalue
        lag = pp.lags
        return stat, pvalue, lag, "OK"
    except Exception as e:
        return np.nan, np.nan, np.nan, str(e)


In [7]:
# ============================================================
# 3. CHẠY UNIT ROOT TEST Ở LEVEL VÀ FIRST DIFFERENCE
# ============================================================

ADF_REGRESSION = "c"
PP_TREND = "c"

# Nếu muốn kiểm định có cả trend, đổi thành:
# ADF_REGRESSION = "ct"
# PP_TREND = "ct"

results = []

# Define ALPHA since it's used in get_order function but not defined in context
ALPHA = 0.05

# Helper functions for order determination and result formatting (assuming they are defined elsewhere or will be added)
def get_order(p_level, p_diff, alpha):
    if p_level < alpha:
        return "I(0)"
    elif p_diff < alpha:
        return "I(1)"
    else:
        return "Check further / possible I(2)"

def format_test_result(stat, pvalue):
    if pd.isna(stat) or pd.isna(pvalue):
        return "N/A"
    stars = ""
    if pvalue < 0.01:
        stars = "***"
    elif pvalue < 0.05:
        stars = "**"
    elif pvalue < 0.1:
        stars = "*"
    return f"{stat:.3f}{stars} ({pvalue:.3f})"

def combine_orders(adf_order, pp_order):
    if adf_order == pp_order:
        return adf_order
    elif "I(0)" in [adf_order, pp_order] and "I(1)" in [adf_order, pp_order]:
        return "Mixed I(0)/I(1)"
    else:
        return "Need further test"

for var in unit_root_vars:
    series_level = data_processed[var].dropna() # Changed df to df_model
    series_diff = series_level.diff().dropna()

    # ADF level
    adf_level_stat, adf_level_p, adf_level_lag, adf_level_note = run_adf(
        series_level,
        regression=ADF_REGRESSION
    )

    # ADF first difference
    adf_diff_stat, adf_diff_p, adf_diff_lag, adf_diff_note = run_adf(
        series_diff,
        regression=ADF_REGRESSION
    )

    # PP level
    pp_level_stat, pp_level_p, pp_level_lag, pp_level_note = run_pp(
        series_level,
        trend=PP_TREND
    )

    # PP first difference
    pp_diff_stat, pp_diff_p, pp_diff_lag, pp_diff_note = run_pp(
        series_diff,
        trend=PP_TREND
    )

    # Xác định bậc tích hợp theo từng test
    adf_order = get_order(adf_level_p, adf_diff_p, ALPHA)
    pp_order = get_order(pp_level_p, pp_diff_p, ALPHA)

    # Kết luận tổng hợp
    final_order = combine_orders(adf_order, pp_order)

    # Trạng thái dùng được cho NARDL hay không
    if final_order in ["I(0)", "I(1)", "Mixed I(0)/I(1)"]:
        nardl_status = "Valid for NARDL"
    elif final_order == "Check further / possible I(2)":
        nardl_status = "Not valid until further check"
    else:
        nardl_status = "Need further test"

    results.append({
        "Variable": var,

        "ADF Level": format_test_result(adf_level_stat, adf_level_p),
        "ADF First Difference": format_test_result(adf_diff_stat, adf_diff_p),
        "ADF Order": adf_order,
        "ADF Lag Level": adf_level_lag,
        "ADF Lag Diff": adf_diff_lag,

        "PP Level": format_test_result(pp_level_stat, pp_level_p),
        "PP First Difference": format_test_result(pp_diff_stat, pp_diff_p),
        "PP Order": pp_order,
        "PP Lag Level": pp_level_lag,
        "PP Lag Diff": pp_diff_lag,

        "Final Order": final_order,
        "NARDL Status": nardl_status
    })


table = pd.DataFrame(results)

print("\n===== TABLE 1. UNIT ROOT TESTS =====")
print(table)


===== TABLE 1. UNIT ROOT TESTS =====
          Variable          ADF Level ADF First Difference ADF Order  \
0           ln_FDI     -0.598 (0.871)    -7.211*** (0.000)      I(1)   
1            ln_TB  -5.103*** (0.000)    -5.929*** (0.000)      I(0)   
2            ln_EX     -1.029 (0.743)     -3.100** (0.027)      I(1)   
3            ln_IM     -1.268 (0.644)     -3.102** (0.026)      I(1)   
4       ln_RER_pos  -7.235*** (0.000)    -5.990*** (0.000)      I(0)   
5       ln_RER_neg  -4.890*** (0.000)    -6.539*** (0.000)      I(0)   
6           IPI_VN     -1.654 (0.455)   -10.631*** (0.000)      I(1)   
7   IPI_World_diff  -6.797*** (0.000)    -6.545*** (0.000)      I(0)   
8           ln_WTI     -2.320 (0.166)    -9.282*** (0.000)      I(1)   
9       ln_M2_diff     -2.027 (0.275)    -4.856*** (0.000)      I(1)   
10           COVID     -1.483 (0.542)   -11.314*** (0.000)      I(1)   

    ADF Lag Level  ADF Lag Diff            PP Level PP First Difference  \
0              10     

In [8]:
# ============================================================
# 4. XUẤT TABLE 3 RA CSV VÀO FOLDER results
# ============================================================
OUTPUT_FILE = os.path.join('../results', "01_unit_root_test.csv")

table.to_csv(OUTPUT_FILE, index=False)